In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/content/World Happiness Report.csv')

In [ ]:
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(df['Country'], df['Happiness Score'], color='skyblue')
plt.xlabel('Country')
plt.ylabel('Happiness Score')
plt.title('Happiness Score by Country')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df['Economy'], df['Happiness Score'], color='green')
plt.xlabel('Economy')
plt.ylabel('Happiness Score')
plt.title('Happiness Score vs Economy')
plt.grid(True)
plt.show()

In [ ]:
factors = ['Economy', 'Family', 'Health', 'Freedom', 'Generosity', 'Corruption', 'Dystopia', 'Job Satisfaction']
bottom = np.zeros(len(df))

plt.figure(figsize=(12, 6))

for factor in factors:
    plt.bar(df['Country'], df[factor], bottom=bottom, label=factor)
    bottom += df[factor].values

plt.xticks(rotation=45)
plt.ylabel('Total Score')
plt.title('Stacked Bar Chart of Happiness Score Factors')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
factors = ['Economy', 'Family', 'Health', 'Freedom', 'Generosity', 'Corruption', 'Dystopia', 'Job Satisfaction']

plt.figure(figsize=(18, 12))

for i, factor in enumerate(factors):
    plt.subplot(3, 3, i + 1)
    plt.scatter(df[factor], df['Happiness Score'], color='teal', alpha=0.7)
    corr = df['Happiness Score'].corr(df[factor])
    plt.xlabel(factor)
    plt.ylabel('Happiness Score')
    plt.title(f'{factor} vs Happiness Score\nCorrelation: {corr:.2f}')
    plt.grid(True)

plt.tight_layout()
plt.show()

Before applying linear regression model there are some NAN values in Job Satisfaction column which need to be resolved first to apply the model. I am using median technique which will not affect the data that much.

In [ ]:
print(df.isna().sum())

In [ ]:
df_cleaned = df.dropna()

In [ ]:
df_cleaned = df.dropna(axis=1)

In [ ]:
df['Job Satisfaction'] = df['Job Satisfaction'].fillna(df['Job Satisfaction'].median())

## Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
X = df[['Economy', 'Family', 'Health', 'Freedom', 'Generosity', 'Corruption', 'Dystopia', 'Job Satisfaction']]
y = df['Happiness Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

**Metric Interpretation**

| Metric | Meaning |
|--------|---------|
| MSE (Mean Squared Error) | Measures how far the predicted values are from actual ones (error). Lower is better. |
| R² Score (Coefficient of Determination) | Shows how much of the variation in happiness score your model explains. Closer to 1 is better. |

| Metric | Bad | Okay | Good | Excellent |
|--------|-----|------|------|-----------|
| **MSE** | > 1.0 | 0.5 – 1.0 | 0.1 – 0.5 | < 0.1 |
| **R² Score** | < 0.5 | 0.5 – 0.7 | 0.7 – 0.9 | > 0.9 |

## Predictive Analysis Based On Input Data Using Trained Linear Regression Model

In [ ]:
def get_country_data():
    country = input("Enter Country Name: ")
    economy = float(input("Economy: "))
    family = float(input("Family: "))
    health = float(input("Health: "))
    freedom = float(input("Freedom: "))
    generosity = float(input("Generosity: "))
    corruption = float(input("Corruption: "))
    dystopia = float(input("Dystopia: "))
    Job_Satisfaction = float(input("Job Satisfaction: "))
    input_data = [[economy, family, health, freedom, generosity, corruption, dystopia, Job_Satisfaction]]
    return country, input_data

In [ ]:
country_name, new_input = get_country_data()
predicted_score = model.predict(new_input)[0]
print(f"\nPredicted Happiness Score for {country_name}: {round(predicted_score, 3)}")

In [ ]:
df['Predicted'] = model.predict(X)
sorted_df = df.sort_values(by='Predicted', ascending=False).reset_index(drop=True)

def get_rank(score, sorted_scores):
    count = sum(score < s for s in sorted_scores)
    return count + 1

rank = get_rank(predicted_score, sorted_df['Predicted'])
print(f"Estimated Happiness Rank: {rank}")

In [ ]:
print(df[['Happiness Score', 'Job Satisfaction']].corr())

We are getting so high accuracy of the model because the correlation of happiness score with job satisfaction and economy is so high that it overfits the model. Either we can remove job satisfaction and economy or we can train a separate model for them.